# 03 — PCA exploration and preprocessing selection

This notebook compares PCA representations for the NIR UCO project.

Main objectives:

1. Load the validated NIR UCO database.
2. Select pure almond and pure peanut objects.
3. Compare matrix representations:
   - `object_mean`
   - `object_median`
   - `balanced_pixels_random`
   - `balanced_pixels_center`
   - optional: `all_pixels`
4. Compare spectral preprocessing methods:
   - raw
   - absorbance
   - SNV / MSC
   - Savitzky-Golay smoothing
   - Savitzky-Golay derivative
   - combined chains
5. Rank preprocessing × matrix combinations using PCA diagnostics.
6. Select candidate preprocessings for downstream SIMCA and MCR analysis.

This notebook is exploratory. It does not select the final SIMCA model.

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
from src.io.database_h5 import load_nir_uco_h5
from src import experiment_config as expcfg

from src.utils import (
    save_parquet,
)

from src.matrices.matrix_registry import build_matrix

from src.spectra.preprocessing_configs import normalize_preprocessing_configs
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.workflows.pca import (
    compare_pca_representations,
)
from src.workflows.pca_selection import (
    add_pca_selection_scores,
    build_pca_scoring_diagnostics,
    select_pca_preprocessing_shortlist,
    validate_pca_preprocessing_shortlist,
)

from src.visualization.plot_pca import (
    plot_explained_variance,
    plot_loadings,
    plot_pca_diagnostic,
    plot_pca_metric_heatmap,
    plot_pca_metric_tradeoff,
    plot_pca_metric_ranking,
)

from src.visualization.plot_scores import (
    plot_scores,
    build_scores_dataframe,
    sample_scores_dataframe,
    plot_scores_density,
    summarize_scores_by_object,
    plot_object_score_summary,
)
from src.visualization.plot_spectra import plot_spectra

%load_ext autoreload
%autoreload 2

In [3]:
# ---------------------------------------------------------------------
# Input database
# ---------------------------------------------------------------------
DB_H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_database.h5"
)

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
# Current workflow:
#   use all non-noisy bands stored in the H5 database.
#
# Later, set USE_WAVELENGTH_WINDOW=True to test a spectral window.
USE_WAVELENGTH_WINDOW = False

WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SUMMARY_PATH = RESULTS_DIR / "pca_summary.parquet"
PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_DIR / "pca_selected_preprocessings.parquet"
PCA_SCORING_DIAGNOSTICS_PATH = RESULTS_DIR / "pca_scoring_diagnostics.parquet"

# ---------------------------------------------------------------------
# PCA data subset
# ---------------------------------------------------------------------
TARGET_CLASS = expcfg.TARGET_CLASS
REFERENCE_CLASSES = expcfg.REFERENCE_CLASSES

# Use pure objects only for PCA exploration.
# This avoids mixing true pure samples with position-reference images.
PCA_SAMPLE_KIND = "pure"

# Use train + validation pure batches; keep pure test batch out of PCA selection.
PCA_ALLOWED_BATCHES = list(expcfg.PCA_ALLOWED_BATCHES)
if PCA_ALLOWED_BATCHES != [1, 2, 3]:
    raise ValueError(f"PCA_ALLOWED_BATCHES must stay [1, 2, 3], got {PCA_ALLOWED_BATCHES}")

# ---------------------------------------------------------------------
# PCA comparison settings
# ---------------------------------------------------------------------
N_COMPONENTS = 20

M_BALANCED_PIXELS = expcfg.M_BALANCED_PIXELS
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
RANDOM_STATE = expcfg.RANDOM_STATE

BALANCED_PIXEL_STRATEGIES = list(expcfg.BALANCED_PIXEL_STRATEGIES)

# Optional because all_pixels can become large.
RUN_ALL_PIXELS = False

# ---------------------------------------------------------------------
# Preprocessing settings
# ---------------------------------------------------------------------
SG_WINDOW_LENGTH = 11
SG_POLYORDER = 2

PREPROCESSING_METHODS = {
    "raw": ("raw",),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "msc": ("msc",),
    "sg_smooth": ("sg_smooth",),
    "sg_d1": ("sg_d1",),
    "sg_d2": ("sg_d2",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_msc": ("absorbance", "msc"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_sg_d2": ("absorbance", "sg_d2"),
    "snv_sg_smooth": ("snv", "sg_smooth"),
    "snv_sg_d1": ("snv", "sg_d1"),
    "snv_sg_d2": ("snv", "sg_d2"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "absorbance_snv_sg_d2": ("absorbance", "snv", "sg_d2"),
}

# ---------------------------------------------------------------------
# Plotting settings
# ---------------------------------------------------------------------
N_TOP_TO_DISPLAY = 20
N_TOP_TO_PLOT = 5
PCA_SELECTION_CONFIG = expcfg.make_pca_selection_config()
MAX_PREPROCESSINGS_PER_MATRIX_FAMILY = PCA_SELECTION_CONFIG.max_preprocessings_per_family
EXPECTED_PCA_MATRIX_FAMILIES = tuple(PCA_SELECTION_CONFIG.expected_families)

RUN_SPECTRA_CHECK_PLOTS = True
RUN_DETAILED_PCA_PLOTS = True
MAX_SPECTRA_TO_PLOT = 50

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("USE_WAVELENGTH_WINDOW:", USE_WAVELENGTH_WINDOW)
print("RESULTS_TAG:", RESULTS_TAG)
print("RUN_ALL_PIXELS:", RUN_ALL_PIXELS)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all
WAVELENGTH_MODE: non_noisy_all
USE_WAVELENGTH_WINDOW: False
RESULTS_TAG: non_noisy_all
RUN_ALL_PIXELS: False


In [4]:
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_object = next(iter(object_db.values()))
    wavelengths = first_object.get("wavelengths")
    wavelengths = np.asarray(wavelengths) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is not None and len(wavelengths) > 0:
    wavelength_config_df = pd.DataFrame([{
        "wavelength_mode": WAVELENGTH_MODE,
        "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
        "results_tag": RESULTS_TAG,
        "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "n_bands": int(len(wavelengths)),
        "min_wavelength_nm": float(np.min(wavelengths)),
        "max_wavelength_nm": float(np.max(wavelengths)),
    }])
else:
    wavelength_config_df = pd.DataFrame([{
        "wavelength_mode": WAVELENGTH_MODE,
        "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
        "results_tag": RESULTS_TAG,
        "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "n_bands": np.nan,
        "min_wavelength_nm": np.nan,
        "max_wavelength_nm": np.nan,
    }])

print("Wavelength configuration:")
display(wavelength_config_df)

if USE_WAVELENGTH_WINDOW:
    print("Detailed wavelength selection:")
    display(wavelength_selection_df)

Wavelength configuration:


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


In [5]:
object_rows = []

for object_id, obj in object_db.items():
    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "image_nut_type": obj.get("image_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
        "is_pure": obj.get("is_pure"),
        "is_mixture": obj.get("is_mixture"),
        "is_position_reference": obj.get("is_position_reference"),
    })

object_meta_df = pd.DataFrame(object_rows)

display(object_meta_df.head())

display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

,object_id,source_clean_key,source_image,sample_kind,object_nut_type,image_nut_type,batch,split,area_pixels,n_pixels,n_bands,is_pure,is_mixture,is_position_reference
0,alm1pea1_obj001,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,84,84,63,False,True,False
1,alm1pea1_obj002,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,73,73,63,False,True,False
2,alm1pea1_obj003,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,91,91,63,False,True,False
3,alm1pea1_obj004,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,73,73,63,False,True,False
4,alm1pea1_obj005,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,126,126,63,False,True,False


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


In [6]:
def subset_object_db_for_pca(
    object_db,
    sample_kind="pure",
    REFERENCE_CLASSES=("almond", "peanut"),
    allowed_batches=None,
):
    """Return a filtered object_db for PCA exploration."""
    out = {}

    REFERENCE_CLASSES = set(map(str, REFERENCE_CLASSES))

    if allowed_batches is not None:
        allowed_batches = set(allowed_batches)

    for object_id, obj in object_db.items():
        if str(obj.get("sample_kind")) != str(sample_kind):
            continue

        if str(obj.get("object_nut_type")) not in REFERENCE_CLASSES:
            continue

        if allowed_batches is not None:
            batch = obj.get("batch")
            if batch not in allowed_batches:
                continue

        out[object_id] = obj

    return out


object_db_pca = subset_object_db_for_pca(
    object_db=object_db,
    sample_kind=PCA_SAMPLE_KIND,
    REFERENCE_CLASSES=REFERENCE_CLASSES,
    allowed_batches=PCA_ALLOWED_BATCHES,
)

print(f"Objects selected for PCA: {len(object_db_pca)}")

pca_object_meta_df = object_meta_df[
    object_meta_df["object_id"].isin(object_db_pca.keys())
].copy()

display(
    pca_object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["object_nut_type", "batch"], na_position="last")
)

if len(object_db_pca) == 0:
    raise RuntimeError("No object selected for PCA. Check filtering parameters.")

Objects selected for PCA: 317


,sample_kind,object_nut_type,batch,n_objects
0,pure,almond,1.0,52
1,pure,almond,2.0,59
2,pure,almond,3.0,55
3,pure,peanut,1.0,46
4,pure,peanut,2.0,52
5,pure,peanut,3.0,53


In [7]:
first_object = next(iter(object_db_pca.values()))
wavelengths = first_object.get("wavelengths")

if wavelengths is not None:
    wavelengths = np.asarray(wavelengths)
    if wavelengths.size == 0:
        wavelengths = None

if wavelengths is None:
    print("No wavelength axis found. PCA plots will use band indices.")
else:
    print("Wavelength axis found.")
    print("n_wavelengths:", len(wavelengths))
    print("first:", wavelengths[:5])
    print("last:", wavelengths[-5:])

Wavelength axis found.
n_wavelengths: 63
first: [ 960.73529412  972.69117647  984.64705882  996.60294118 1008.55882353]
last: [1654.17647059 1666.13235294 1678.08823529 1690.04411765 1702.        ]


In [8]:
X_mean_raw, y_mean_raw, meta_mean_raw = build_matrix(
    object_db=object_db_pca,
    matrix_method="object_mean",
    filters={
        "object_nut_type": list(REFERENCE_CLASSES),
    },
)

meta_mean_raw_df = pd.DataFrame(meta_mean_raw)

print("X_mean_raw:", X_mean_raw.shape)
print("y_mean_raw:", y_mean_raw.shape)
display(meta_mean_raw_df.head())

# Optional sampling for detailed plot.
if RUN_SPECTRA_CHECK_PLOTS:
    # Optional sampling for detailed plot.
    rng = np.random.default_rng(RANDOM_STATE)

    if X_mean_raw.shape[0] > MAX_SPECTRA_TO_PLOT:
        sampled_indices = []

        temp_df = meta_mean_raw_df.copy()
        temp_df["_row_index"] = np.arange(len(temp_df))
        temp_df["label"] = y_mean_raw

        for label, sub in temp_df.groupby("label", dropna=False):
            n = min(MAX_SPECTRA_TO_PLOT // len(REFERENCE_CLASSES), len(sub))
            sampled = sub.sample(n=n, random_state=RANDOM_STATE)
            sampled_indices.extend(sampled["_row_index"].tolist())

        sampled_indices = sorted(sampled_indices)
    else:
        sampled_indices = np.arange(X_mean_raw.shape[0])

    plot_spectra(
        X_mean_raw[sampled_indices],
        wavelengths=wavelengths,
        labels=y_mean_raw[sampled_indices],
        names=meta_mean_raw_df.iloc[sampled_indices]["object_id"].to_numpy()
            if "object_id" in meta_mean_raw_df.columns
            else None,
        reducer="none",
        title="Raw object mean spectra — sampled pure objects",
        y_title="Reflectance",
        show=True,
    )

    plot_spectra(
        X_mean_raw,
        wavelengths=wavelengths,
        labels=y_mean_raw,
        reducer="mean_std",
        title="Raw object mean spectra — mean ± std by class",
        y_title="Reflectance",
        show=True,
    )
else:
    print("Spectral check plots skipped.")

X_mean_raw: (317, 63)
y_mean_raw: (317,)


,object_id,label,source_image,source_clean_key,source_image_id,batch,area,sample_kind
0,almond1_obj001,almond,almond1,almond1,almond1_sb,1,54,pure
1,almond1_obj002,almond,almond1,almond1,almond1_sb,1,95,pure
2,almond1_obj003,almond,almond1,almond1,almond1_sb,1,52,pure
3,almond1_obj004,almond,almond1,almond1,almond1_sb,1,98,pure
4,almond1_obj005,almond,almond1,almond1,almond1_sb,1,84,pure


## 1. PCA comparison design

We compare several matrix representations.

To avoid mixing pure samples with position-reference images, the PCA input object database has already been filtered to pure almond and pure peanut objects only.

Because the corrected database currently stores all objects with `split="projection"` for backward compatibility with the previous script, this notebook does not filter by split.

In [9]:
pca_runs = [
    {
        "run_id": "object_matrices",
        "matrix_methods": ["object_mean", "object_median"],
        "balanced_pixel_strategy": "random",
    },
    {
        "run_id": "balanced_pixels_random",
        "matrix_methods": ["balanced_pixels"],
        "balanced_pixel_strategy": "random",
    },
]

if "center" in BALANCED_PIXEL_STRATEGIES:
    pca_runs.append({
        "run_id": "balanced_pixels_center",
        "matrix_methods": ["balanced_pixels"],
        "balanced_pixel_strategy": "center",
    })

if RUN_ALL_PIXELS:
    pca_runs.append({
        "run_id": "all_pixels",
        "matrix_methods": ["all_pixels"],
        "balanced_pixel_strategy": "random",
    })

pca_runs_df = pd.DataFrame(pca_runs)
pca_runs_df

,run_id,matrix_methods,balanced_pixel_strategy
0,object_matrices,"[object_mean, object_median]",random
1,balanced_pixels_random,[balanced_pixels],random
2,balanced_pixels_center,[balanced_pixels],center


In [10]:
valid_preprocessing_configs = normalize_preprocessing_configs(PREPROCESSING_METHODS)
valid_preprocessing_configs

{'raw': ('raw',),
 'absorbance': ('absorbance',),
 'snv': ('snv',),
 'msc': ('msc',),
 'sg_smooth': ('sg_smooth',),
 'sg_d1': ('sg_d1',),
 'sg_d2': ('sg_d2',),
 'absorbance_snv': ('absorbance', 'snv'),
 'absorbance_msc': ('absorbance', 'msc'),
 'absorbance_sg_smooth': ('absorbance', 'sg_smooth'),
 'absorbance_sg_d1': ('absorbance', 'sg_d1'),
 'absorbance_sg_d2': ('absorbance', 'sg_d2'),
 'snv_sg_smooth': ('snv', 'sg_smooth'),
 'snv_sg_d1': ('snv', 'sg_d1'),
 'snv_sg_d2': ('snv', 'sg_d2'),
 'absorbance_snv_sg_smooth': ('absorbance', 'snv', 'sg_smooth'),
 'absorbance_snv_sg_d1': ('absorbance', 'snv', 'sg_d1'),
 'absorbance_snv_sg_d2': ('absorbance', 'snv', 'sg_d2')}

In [11]:
pca_summary_parts = []
pca_results_registry = {}

for run in pca_runs:
    run_id = run["run_id"]
    matrix_methods = run["matrix_methods"]
    balanced_pixel_strategy = run["balanced_pixel_strategy"]

    print("=" * 100)
    print("PCA run:", run_id)
    print("matrix_methods:", matrix_methods)
    print("balanced_pixel_strategy:", balanced_pixel_strategy)
    print("=" * 100)

    try:
        summary_run_df, results_run = compare_pca_representations(
            object_db=object_db_pca,
            matrix_methods=matrix_methods,
            preprocessing_methods=valid_preprocessing_configs,
            allowed_splits=None,  # Important: the database was built with forced split="projection"
            allowed_labels=REFERENCE_CLASSES,
            n_components=N_COMPONENTS,
            m=M_BALANCED_PIXELS,
            wavelengths=wavelengths,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            sg_window_length=SG_WINDOW_LENGTH,
            sg_polyorder=SG_POLYORDER,
            balanced_pixel_strategy=balanced_pixel_strategy,
        )

        summary_run_df["run_id"] = run_id

        # Backward compatibility if an old version of src.workflows.pca is loaded.
        if "balanced_pixel_strategy" not in summary_run_df.columns:
            summary_run_df["balanced_pixel_strategy"] = np.where(
                summary_run_df["matrix_method"].eq("balanced_pixels"),
                balanced_pixel_strategy,
                "not_applicable",
            )
        if "balanced_pixel_strategy_effective" not in summary_run_df.columns:
            summary_run_df["balanced_pixel_strategy_effective"] = balanced_pixel_strategy
        if "matrix_family" not in summary_run_df.columns:
            summary_run_df["matrix_family"] = np.where(
                summary_run_df["matrix_method"].isin(["object_mean", "object_median"]),
                "object_matrix",
                "pixel_matrix",
            )
        if "matrix_variant" not in summary_run_df.columns:
            summary_run_df["matrix_variant"] = np.where(
                summary_run_df["matrix_method"].eq("balanced_pixels"),
                "balanced_pixels_" + summary_run_df["balanced_pixel_strategy_effective"].astype(str),
                summary_run_df["matrix_method"].astype(str),
            )

        pca_summary_parts.append(summary_run_df)
        pca_results_registry[run_id] = results_run

    except Exception as exc:
        print(f"[ERROR] PCA run failed: {run_id}")
        print(repr(exc))
        raise

pca_summary_df = pd.concat(pca_summary_parts, ignore_index=True)

display(pca_summary_df.head())
print("PCA summary shape:", pca_summary_df.shape)

PCA run: object_matrices
matrix_methods: ['object_mean', 'object_median']
balanced_pixel_strategy: random

=== Matrix method: object_mean ===
X shape: (317, 63)
Labels: {'almond': 166, 'peanut': 151}
  - preprocessing: raw
  - preprocessing: absorbance
  - preprocessing: snv
  - preprocessing: msc
  - preprocessing: sg_smooth
  - preprocessing: sg_d1
  - preprocessing: sg_d2
  - preprocessing: absorbance_snv
  - preprocessing: absorbance_msc
  - preprocessing: absorbance_sg_smooth
  - preprocessing: absorbance_sg_d1
  - preprocessing: absorbance_sg_d2
  - preprocessing: snv_sg_smooth
  - preprocessing: snv_sg_d1
  - preprocessing: snv_sg_d2
  - preprocessing: absorbance_snv_sg_smooth
  - preprocessing: absorbance_snv_sg_d1
  - preprocessing: absorbance_snv_sg_d2

=== Matrix method: object_median ===
X shape: (317, 63)
Labels: {'almond': 166, 'peanut': 151}
  - preprocessing: raw
  - preprocessing: absorbance
  - preprocessing: snv
  - preprocessing: msc
  - preprocessing: sg_smooth
  -

,matrix_family,matrix_variant,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id
0,object_matrix,object_mean,not_applicable,random,object_mean,snv_sg_smooth,snv+sg_smooth,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.435213,0.351317,0.112835,0.786531,0.899366,0.337792,0.669172,0.400779,1.433219,1.694245,4.521109,4,4,0.314925,0.033345,9.444572,0.017027,0.011048,0.061547,2.990536,2.550148,6.427924,NaN,NaN,NaN,NaN,166,151,object_matrices
1,object_matrix,object_median,not_applicable,random,object_median,snv_sg_smooth,snv+sg_smooth,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.428816,0.280304,0.119524,0.709120,0.828645,0.343511,0.641684,0.037551,1.141485,1.189302,2.577439,5,6,0.226484,0.022855,9.909674,0.046012,0.028737,0.137929,2.990536,2.329076,7.864888,NaN,NaN,NaN,NaN,166,151,object_matrices
2,object_matrix,object_median,not_applicable,random,object_median,snv,snv,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.379209,0.252221,0.103677,0.631430,0.735107,0.353341,0.611606,0.057882,1.140709,1.191818,2.581632,10,21,0.222377,0.022271,9.984837,0.085850,0.064123,0.210698,2.990536,2.313024,7.758865,NaN,NaN,NaN,NaN,166,151,object_matrices
3,object_matrix,object_median,not_applicable,random,object_median,msc,msc,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.379769,0.252652,0.103522,0.632421,0.735943,0.031109,0.610478,0.057130,1.136096,1.188856,2.563718,10,21,0.221618,0.022278,9.947966,0.000664,0.000493,0.001629,2.990536,2.301725,7.749371,NaN,NaN,NaN,NaN,166,151,object_matrices
4,object_matrix,object_mean,not_applicable,random,object_mean,snv,snv,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.422100,0.341429,0.113928,0.763529,0.877457,0.341179,0.504780,0.512179,1.584932,1.651803,4.736707,4,5,0.305837,0.032961,9.278904,0.022946,0.015877,0.077726,2.990536,2.539259,6.385352,NaN,NaN,NaN,NaN,166,151,object_matrices


PCA summary shape: (72, 42)


In [12]:
pca_scored_df = add_pca_selection_scores(
    pca_summary_df,
    config=PCA_SELECTION_CONFIG,
)

pca_scored_df = (
    pca_scored_df
    .sort_values(
        ["selection_score", "selection_score_stability_std"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

if "rank" in pca_scored_df.columns:
    pca_scored_df = pca_scored_df.drop(columns=["rank"])
pca_scored_df.insert(0, "rank", np.arange(1, len(pca_scored_df) + 1))

pca_scoring_diagnostics_df = build_pca_scoring_diagnostics(
    pca_scored_df,
    config=PCA_SELECTION_CONFIG,
)

score_diagnostic_cols = [
    "rank",
    "matrix_family",
    "matrix_variant",
    "preprocessing",
    "selection_score",
    "selection_score_without_stability",
    "object_matrix_score",
    "pixel_matrix_score",
    "selection_score_stability_penalty",
    "selection_score_stability_std",
    "selection_score_rank_std",
    "selection_flag",
    "pca_validation_warning",
]
score_diagnostic_cols = [col for col in score_diagnostic_cols if col in pca_scoring_diagnostics_df.columns]

display(pca_scoring_diagnostics_df[score_diagnostic_cols].head(N_TOP_TO_DISPLAY))

,rank,matrix_family,matrix_variant,preprocessing,selection_score,selection_score_without_stability,object_matrix_score,pixel_matrix_score,selection_score_stability_penalty,selection_score_stability_std,selection_score_rank_std,selection_flag,pca_validation_warning
0,1,pixel_matrix,balanced_pixels_random,snv_sg_smooth,5.823665,6.874581,NaN,5.823665,1.050916,4.203665,2.496462,score_unstable,score_unstable
1,2,pixel_matrix,balanced_pixels_random,absorbance_snv_sg_smooth,5.254788,6.143726,NaN,5.254788,0.888938,3.555753,2.368650,candidate,
2,3,pixel_matrix,balanced_pixels_random,snv,5.047769,6.023611,NaN,5.047769,0.975842,3.903368,3.308330,score_unstable,score_unstable
3,4,pixel_matrix,balanced_pixels_center,snv_sg_smooth,4.289968,6.142479,NaN,4.289968,1.852511,7.410044,2.389603,batch_sensitive,batch_sensitive; score_unstable
4,5,pixel_matrix,balanced_pixels_center,absorbance_snv_sg_smooth,3.413084,5.351800,NaN,3.413084,1.938716,7.754863,3.830025,batch_sensitive,batch_sensitive; score_unstable
5,6,pixel_matrix,balanced_pixels_center,sg_smooth,2.797507,3.126480,NaN,2.797507,0.328973,1.315891,3.747848,candidate,
6,7,pixel_matrix,balanced_pixels_center,raw,2.794307,3.123342,NaN,2.794307,0.329035,1.316140,3.741266,candidate,
7,8,pixel_matrix,balanced_pixels_random,snv_sg_d1,2.761931,3.056006,NaN,2.761931,0.294075,1.176300,1.500707,candidate,
8,9,object_matrix,object_mean,absorbance_sg_d1,2.568244,2.896111,2.568244,NaN,0.327867,1.311469,3.476966,candidate,
9,10,pixel_matrix,balanced_pixels_center,snv_sg_d1,2.412894,3.427935,NaN,2.412894,1.015041,4.060164,1.645655,batch_sensitive,batch_sensitive; score_unstable


In [13]:
pca_scored_df.columns

Index(['rank', 'matrix_family', 'matrix_variant', 'balanced_pixel_strategy',
       'balanced_pixel_strategy_effective', 'matrix_method', 'preprocessing',
       'preprocessing_steps', 'n_observations', 'n_bands', 'n_components', 'm',
       'm_effective', 'label_counts', 'evr_pc1', 'evr_pc2', 'evr_pc3',
       'cum_pc2', 'cum_pc3', 'centroid_distance_pc1_pc2', 'fisher_pc1',
       'fisher_pc2', 'fisher_pc3', 'mahalanobis_pc1_pc2',
       'mahalanobis_pc1_pc2_pc3', 'ncomp_90', 'ncomp_95', 'class_trace_ratio',
       'batch_trace_ratio', 'class_over_batch_ratio', 'train_q_mean',
       'train_q_median', 'train_q_q95', 'train_t2_mean', 'train_t2_median',
       'train_t2_q95', 'object_class_trace_ratio', 'object_batch_trace_ratio',
       'mean_intra_object_trace', 'object_over_intra_ratio', 'n_label_almond',
       'n_label_peanut', 'run_id', 'object_matrix_score_raw',
       'object_matrix_score', 'pixel_matrix_score_raw', 'pixel_matrix_score',
       'contrib_plus_class_trace_ratio',


In [14]:
print("pca_summary_df columns:")
print(sorted(pca_summary_df.columns))

print("\npca_scored_df columns:")
print(sorted(pca_scored_df.columns))

assert "matrix_variant" in pca_summary_df.columns
assert "matrix_variant" in pca_scored_df.columns
assert pca_scored_df["matrix_variant"].notna().all()

pca_scored_df[
    ["matrix_family", "matrix_method", "balanced_pixel_strategy", "matrix_variant"]
].drop_duplicates()

pca_summary_df columns:
['balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'batch_trace_ratio', 'centroid_distance_pc1_pc2', 'class_over_batch_ratio', 'class_trace_ratio', 'cum_pc2', 'cum_pc3', 'evr_pc1', 'evr_pc2', 'evr_pc3', 'fisher_pc1', 'fisher_pc2', 'fisher_pc3', 'label_counts', 'm', 'm_effective', 'mahalanobis_pc1_pc2', 'mahalanobis_pc1_pc2_pc3', 'matrix_family', 'matrix_method', 'matrix_variant', 'mean_intra_object_trace', 'n_bands', 'n_components', 'n_label_almond', 'n_label_peanut', 'n_observations', 'ncomp_90', 'ncomp_95', 'object_batch_trace_ratio', 'object_class_trace_ratio', 'object_over_intra_ratio', 'preprocessing', 'preprocessing_steps', 'run_id', 'train_q_mean', 'train_q_median', 'train_q_q95', 'train_t2_mean', 'train_t2_median', 'train_t2_q95']

pca_scored_df columns:
['active_family_score_raw', 'balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'batch_high_threshold', 'batch_trace_ratio', 'centroid_distance_pc1_pc2', 'class_over_batch_ratio

,matrix_family,matrix_method,balanced_pixel_strategy,matrix_variant
0,pixel_matrix,balanced_pixels,random,balanced_pixels_random
3,pixel_matrix,balanced_pixels,center,balanced_pixels_center
8,object_matrix,object_mean,not_applicable,object_mean
16,object_matrix,object_median,not_applicable,object_median


In [15]:
# The scored table is the useful PCA summary for downstream notebooks.
# The diagnostic table keeps only score inputs, contributions, stability, and flags.
save_parquet(pca_scored_df, PCA_SUMMARY_PATH)
save_parquet(pca_scoring_diagnostics_df, PCA_SCORING_DIAGNOSTICS_PATH)

print("Saved PCA summary:")
print(" -", PCA_SUMMARY_PATH)
print("Saved PCA scoring diagnostics:")
print(" -", PCA_SCORING_DIAGNOSTICS_PATH)

Saved PCA summary:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_summary.parquet
Saved PCA scoring diagnostics:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_scoring_diagnostics.parquet


## 2. Global PCA ranking

We now inspect the best preprocessing × matrix combinations.

Useful interpretation:

- High class separation is good.
- High batch effect is undesirable.
- For pixel matrices, object-level separation is more important than raw pixel-level separation.
- The final choice should not rely on one metric only.

In [16]:
ranking_cols = [
    "rank",
    "matrix_family",
    "matrix_variant",
    "matrix_method",
    "balanced_pixel_strategy",
    "preprocessing",
    "selection_score",
    "selection_flag",
    "n_observations",
    "n_label_almond",
    "n_label_peanut",
    "label_counts",
    "evr_pc1",
    "cum_pc3",
    "ncomp_90",
    "ncomp_95",
    "fisher_pc1",
    "fisher_pc2",
    "mahalanobis_pc1_pc2_pc3",
    "class_trace_ratio",
    "batch_trace_ratio",
    "class_over_batch_ratio",
    "object_class_trace_ratio",
    "object_batch_trace_ratio",
    "object_over_intra_ratio",
    "mean_intra_object_trace",
]

top_candidates_df = pca_scored_df.copy()
#top_candidates_df.insert(0, "rank", np.arange(1, len(top_candidates_df) + 1))

available_ranking_cols = [col for col in ranking_cols if col in top_candidates_df.columns]

top_candidates_df[available_ranking_cols].head(N_TOP_TO_DISPLAY)

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,1,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_smooth,5.823665,score_unstable,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.662961,0.873493,4,4,0.006142,0.205473,0.728653,0.017140,0.003037,5.643716,0.192140,0.027821,0.118025,1.115901e+00
1,2,pixel_matrix,balanced_pixels_random,balanced_pixels,random,absorbance_snv_sg_smooth,5.254788,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.527433,0.872764,4,5,0.004100,0.083334,0.518073,0.014955,0.002999,4.986508,0.133514,0.023049,0.143751,1.178420e+00
2,3,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv,5.047769,score_unstable,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.593139,0.790372,5,11,0.006124,0.157923,0.632313,0.014183,0.003171,4.473190,0.146186,0.027700,0.124477,1.172803e+00
3,4,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_smooth,4.289968,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.581754,0.848808,4,5,0.025697,0.271131,0.822572,0.032668,0.008997,3.631048,0.143496,0.034588,0.331167,5.470031e-01
4,5,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_snv_sg_smooth,3.413084,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.496550,0.855052,4,5,0.035182,0.087680,0.740983,0.030968,0.009235,3.353289,0.154354,0.040382,0.284407,6.828924e-01
5,6,pixel_matrix,balanced_pixels_center,balanced_pixels,center,sg_smooth,2.797507,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.962823,0.997545,1,1,0.024985,0.044919,0.404735,0.012795,0.012667,1.010087,0.022814,0.025791,1.072383,3.698165e-01
6,7,pixel_matrix,balanced_pixels_center,balanced_pixels,center,raw,2.794307,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.961069,0.996216,1,1,0.024966,0.045039,0.383306,0.012782,0.012671,1.008699,0.022789,0.025799,1.072405,3.704175e-01
7,8,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_d1,2.761931,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.811937,0.948484,2,4,0.015267,0.068132,0.410924,0.009769,0.003858,2.532125,0.061123,0.021209,0.213182,4.662923e-04
8,9,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,2.568244,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.769020,0.965067,2,3,0.037632,0.448945,0.663760,0.053277,0.004290,12.418964,NaN,NaN,NaN,NaN
9,10,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_d1,2.412894,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.797545,0.941946,3,4,0.036948,0.055188,0.439840,0.018456,0.008549,2.158910,0.070788,0.029022,0.387521,2.803274e-04


In [17]:
top_candidates_objects_df = top_candidates_df[top_candidates_df['matrix_method'].str.startswith('object')]
top_candidates_objects_df[available_ranking_cols]

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
8,9,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,2.568244,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.769020,0.965067,2,3,0.037632,0.448945,0.663760,0.053277,0.004290,12.418964,NaN,NaN,NaN,NaN
14,15,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d2,2.169377,weak_relative_separation,317,166,151,"{'almond': 166, 'peanut': 151}",0.860111,0.973547,2,3,0.010468,0.445031,0.024193,0.028776,0.003217,8.944332,NaN,NaN,NaN,NaN
16,17,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d1,1.972732,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.712302,0.949862,2,4,0.075230,0.510832,0.776690,0.075748,0.003593,21.080556,NaN,NaN,NaN,NaN
17,18,object_matrix,object_median,object_median,not_applicable,absorbance_snv_sg_d2,1.964249,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.884374,0.950063,2,3,0.215445,0.043348,0.364598,0.100973,0.005256,19.209220,NaN,NaN,NaN,NaN
18,19,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_d2,1.907236,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.882689,0.976093,2,3,0.109921,0.044753,0.248571,0.071318,0.009276,7.688139,NaN,NaN,NaN,NaN
27,28,object_matrix,object_median,object_median,not_applicable,snv_sg_d2,1.300178,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.754498,0.917561,3,5,0.273373,0.031989,0.301613,0.111641,0.008168,13.668087,NaN,NaN,NaN,NaN
28,29,object_matrix,object_mean,object_mean,not_applicable,snv_sg_d2,1.047741,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.788379,0.954863,2,3,0.128031,0.011865,0.189940,0.076905,0.015338,5.013878,NaN,NaN,NaN,NaN
29,30,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d2,0.975399,weak_relative_separation,317,166,151,"{'almond': 166, 'peanut': 151}",0.800884,0.942751,2,4,0.000342,0.573878,0.030791,0.032776,0.003075,10.657487,NaN,NaN,NaN,NaN
31,32,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_d1,0.932149,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.833440,0.966479,2,3,0.149999,0.925057,3.450543,0.122692,0.021761,5.638177,NaN,NaN,NaN,NaN
32,33,object_matrix,object_median,object_median,not_applicable,absorbance_snv_sg_smooth,0.827933,score_unstable,317,166,151,"{'almond': 166, 'peanut': 151}",0.515113,0.859508,4,5,0.390359,0.988409,2.826110,0.267170,0.024087,11.092096,NaN,NaN,NaN,NaN


In [18]:
top_candidates_pixels_df = top_candidates_df[~top_candidates_df['matrix_method'].str.startswith('object')]
top_candidates_pixels_df[available_ranking_cols]

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,1,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_smooth,5.823665,score_unstable,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.662961,0.873493,4,4,0.006142,0.205473,0.728653,0.017140,0.003037,5.643716,0.192140,0.027821,0.118025,1.115901e+00
1,2,pixel_matrix,balanced_pixels_random,balanced_pixels,random,absorbance_snv_sg_smooth,5.254788,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.527433,0.872764,4,5,0.004100,0.083334,0.518073,0.014955,0.002999,4.986508,0.133514,0.023049,0.143751,1.178420e+00
2,3,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv,5.047769,score_unstable,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.593139,0.790372,5,11,0.006124,0.157923,0.632313,0.014183,0.003171,4.473190,0.146186,0.027700,0.124477,1.172803e+00
3,4,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_smooth,4.289968,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.581754,0.848808,4,5,0.025697,0.271131,0.822572,0.032668,0.008997,3.631048,0.143496,0.034588,0.331167,5.470031e-01
4,5,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_snv_sg_smooth,3.413084,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.496550,0.855052,4,5,0.035182,0.087680,0.740983,0.030968,0.009235,3.353289,0.154354,0.040382,0.284407,6.828924e-01
5,6,pixel_matrix,balanced_pixels_center,balanced_pixels,center,sg_smooth,2.797507,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.962823,0.997545,1,1,0.024985,0.044919,0.404735,0.012795,0.012667,1.010087,0.022814,0.025791,1.072383,3.698165e-01
6,7,pixel_matrix,balanced_pixels_center,balanced_pixels,center,raw,2.794307,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.961069,0.996216,1,1,0.024966,0.045039,0.383306,0.012782,0.012671,1.008699,0.022789,0.025799,1.072405,3.704175e-01
7,8,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_d1,2.761931,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.811937,0.948484,2,4,0.015267,0.068132,0.410924,0.009769,0.003858,2.532125,0.061123,0.021209,0.213182,4.662923e-04
9,10,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_d1,2.412894,batch_sensitive,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.797545,0.941946,3,4,0.036948,0.055188,0.439840,0.018456,0.008549,2.158910,0.070788,0.029022,0.387521,2.803274e-04
10,11,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_d2,2.411545,candidate,12360,6494,5866,"{'almond': 6494, 'peanut': 5866}",0.864963,0.964799,2,3,0.018419,0.001516,0.133269,0.008302,0.002457,3.378791,0.059557,0.015725,0.169383,9.214586e-07


In [19]:
plot_pca_metric_ranking(
    top_candidates_objects_df,
    metric="selection_score",
    group_col="matrix_method",
    label_col="preprocessing",
    ascending=False,
    top_n=20,
    title="PCA selection score ranking by matrix representation",
    show=True,
)

In [20]:
plot_pca_metric_ranking(
    top_candidates_pixels_df,
    metric="selection_score",
    group_col="matrix_variant",
    label_col="preprocessing",
    ascending=False,
    top_n=20,
    title="PCA selection score ranking by matrix representation",
    show=True,
)

In [21]:
plot_pca_metric_heatmap(
    pca_scored_df,
    metric="class_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_variant",
    title="PCA class trace ratio — preprocessing × matrix representation",
    show=True,
)

In [22]:
plot_pca_metric_heatmap(
    pca_scored_df,
    metric="batch_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_variant",
    title="PCA batch trace ratio — preprocessing × matrix representation",
    show=True,
)

In [23]:
plot_pca_metric_tradeoff(
    pca_scored_df,
    x_metric="batch_trace_ratio",
    y_metric="class_trace_ratio",
    color_by="matrix_variant",
    symbol_by=None,
    show_pareto=True,
    label_col="preprocessing",
    label_top_n=8,
    title="PCA trade-off — class separation vs batch effect",
    show=True,
)

In [24]:
pixel_metric_df = pca_scored_df[
    pca_scored_df["matrix_method"].isin(["balanced_pixels", "all_pixels"])
].copy()

if len(pixel_metric_df) > 0:
    plot_pca_metric_tradeoff(
        pixel_metric_df,
        x_metric="object_batch_trace_ratio",
        y_metric="object_class_trace_ratio",
        color_by="matrix_variant",
        symbol_by=None,
        show_pareto=True,
        label_col="preprocessing",
        label_top_n=8,
        size_by="object_over_intra_ratio",
        title="Pixel matrix trade-off — object class separation vs object batch effect",
        show=True,
    )
else:
    print("No pixel-level matrix result available.")

In [25]:
def get_pca_result_from_row(row, registry):
    """
    Retrieve full PCA result from one summary row.
    """
    run_id = row["run_id"]
    matrix_method = row["matrix_method"]
    preprocessing = row["preprocessing"]

    return registry[run_id][matrix_method][preprocessing]


def metadata_value(metadata, *possible_keys, default=None):
    """
    Robustly retrieve one metadata array from possible keys.
    """
    for key in possible_keys:
        if key in metadata and metadata[key] is not None:
            return metadata[key]
    return default

In [26]:
best_by_variant_pixels_df = top_candidates_pixels_df.sort_values("selection_score", ascending=False).groupby("matrix_variant", as_index=False).first().reset_index(drop=True)
best_by_variant_pixels_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,object_matrix_score_raw,object_matrix_score,pixel_matrix_score_raw,pixel_matrix_score,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,active_family_score_raw,selection_score_without_stability,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_score_stability_mean,selection_score_stability_std,selection_score_rank_std,selection_score_stability_penalty,separation_low_threshold,batch_high_threshold,projection_high_threshold,validation_shift_high_threshold,score_stability_high_threshold,selection_flag,pca_validation_warning,pca_validation_pass
0,balanced_pixels_center,4,pixel_matrix,center,center,balanced_pixels,snv_sg_smooth,snv+sg_smooth,12360,63,20,40.0,40.0,"{'almond': 6494, 'peanut': 5866}",0.581754,0.151187,0.115867,0.732941,0.848808,0.286511,0.025697,0.271131,0.024626,0.781577,0.822572,4,5,0.032668,0.008997,3.631048,0.121646,0.049876,0.217524,2.999757,1.459400,8.492593,0.143496,0.034588,0.547003,0.331167,6494,5866,balanced_pixels_center,NaN,NaN,6.142479,4.289968,NaN,NaN,NaN,-0.0,-0.0,-0.272727,6.142479,6.142479,4.289968,6.933929,0.401533,-0.653120,-0.267136,6.472231,7.410044,2.389603,1.852511,0.017752,0.028918,NaN,NaN,3.785244,batch_sensitive,batch_sensitive; score_unstable,False
1,balanced_pixels_random,1,pixel_matrix,random,random,balanced_pixels,snv_sg_smooth,snv+sg_smooth,12360,63,20,40.0,40.0,"{'almond': 6494, 'peanut': 5866}",0.662961,0.114969,0.095564,0.777930,0.873493,0.265611,0.006142,0.205473,0.041886,0.653878,0.728653,4,4,0.017140,0.003037,5.643716,0.174507,0.101064,0.538244,2.999757,0.834746,8.904764,0.192140,0.027821,1.115901,0.118025,6494,5866,balanced_pixels_random,NaN,NaN,6.874581,5.823665,NaN,NaN,NaN,-0.0,-0.0,-0.163636,6.874581,6.874581,5.823665,8.499360,0.027473,-0.970353,-0.518262,6.666354,4.203665,2.496462,1.050916,0.017752,0.028918,NaN,NaN,3.785244,score_unstable,score_unstable,False


In [27]:
best_by_variant_objects_df = top_candidates_objects_df.sort_values("selection_score", ascending=False).groupby("matrix_variant", as_index=False).first().reset_index(drop=True)
best_by_variant_objects_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,object_matrix_score_raw,object_matrix_score,pixel_matrix_score_raw,pixel_matrix_score,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,active_family_score_raw,selection_score_without_stability,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_score_stability_mean,selection_score_stability_std,selection_score_rank_std,selection_score_stability_penalty,separation_low_threshold,batch_high_threshold,projection_high_threshold,validation_shift_high_threshold,score_stability_high_threshold,selection_flag,pca_validation_warning,pca_validation_pass
0,object_mean,9,object_matrix,not_applicable,random,object_mean,absorbance_sg_d1,absorbance+sg_d1,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.769020,0.155861,0.040186,0.924881,0.965067,0.000788,0.037632,0.448945,0.365638,0.587939,0.66376,2,3,0.053277,0.004290,12.418964,1.269160e-07,8.470339e-08,3.794666e-07,2.990536,2.114916,7.921218,NaN,NaN,NaN,NaN,166,151,object_matrices,2.896111,2.568244,NaN,NaN,-0.231910,-0.012698,3.140719,-0.0,-0.0,-0.00,2.896111,2.896111,2.568244,NaN,NaN,NaN,NaN,2.203837,1.311469,3.476966,0.327867,0.04141,0.024814,NaN,NaN,3.88622,candidate,,True
1,object_median,17,object_matrix,not_applicable,random,object_median,absorbance_sg_d1,absorbance+sg_d1,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.712302,0.196013,0.041547,0.908315,0.949862,0.001114,0.075230,0.510832,0.074890,0.756382,0.77669,2,4,0.075748,0.003593,21.080556,2.373491e-07,1.859644e-07,5.793017e-07,2.990536,2.067967,8.295867,NaN,NaN,NaN,NaN,166,151,object_matrices,2.591736,1.972732,NaN,NaN,-0.531156,-0.028871,3.031763,-0.0,-0.0,0.12,2.591736,2.591736,1.972732,NaN,NaN,NaN,NaN,2.524727,2.476014,3.511525,0.619003,0.04141,0.024814,NaN,NaN,3.88622,candidate,,True


In [28]:
best_by_variant_df = pd.concat([best_by_variant_objects_df, best_by_variant_pixels_df], ignore_index=True)
best_by_variant_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,object_matrix_score_raw,object_matrix_score,pixel_matrix_score_raw,pixel_matrix_score,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,active_family_score_raw,selection_score_without_stability,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_score_stability_mean,selection_score_stability_std,selection_score_rank_std,selection_score_stability_penalty,separation_low_threshold,batch_high_threshold,projection_high_threshold,validation_shift_high_threshold,score_stability_high_threshold,selection_flag,pca_validation_warning,pca_validation_pass
0,object_mean,9,object_matrix,not_applicable,random,object_mean,absorbance_sg_d1,absorbance+sg_d1,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.769020,0.155861,0.040186,0.924881,0.965067,0.000788,0.037632,0.448945,0.365638,0.587939,0.663760,2,3,0.053277,0.004290,12.418964,1.269160e-07,8.470339e-08,3.794666e-07,2.990536,2.114916,7.921218,NaN,NaN,NaN,NaN,166,151,object_matrices,2.896111,2.568244,NaN,NaN,-0.231910,-0.012698,3.140719,-0.0,-0.0,-0.000000,2.896111,2.896111,2.568244,NaN,NaN,NaN,NaN,2.203837,1.311469,3.476966,0.327867,0.041410,0.024814,NaN,NaN,3.886220,candidate,,True
1,object_median,17,object_matrix,not_applicable,random,object_median,absorbance_sg_d1,absorbance+sg_d1,317,63,20,NaN,NaN,"{'almond': 166, 'peanut': 151}",0.712302,0.196013,0.041547,0.908315,0.949862,0.001114,0.075230,0.510832,0.074890,0.756382,0.776690,2,4,0.075748,0.003593,21.080556,2.373491e-07,1.859644e-07,5.793017e-07,2.990536,2.067967,8.295867,NaN,NaN,NaN,NaN,166,151,object_matrices,2.591736,1.972732,NaN,NaN,-0.531156,-0.028871,3.031763,-0.0,-0.0,0.120000,2.591736,2.591736,1.972732,NaN,NaN,NaN,NaN,2.524727,2.476014,3.511525,0.619003,0.041410,0.024814,NaN,NaN,3.886220,candidate,,True
2,balanced_pixels_center,4,pixel_matrix,center,center,balanced_pixels,snv_sg_smooth,snv+sg_smooth,12360,63,20,40.0,40.0,"{'almond': 6494, 'peanut': 5866}",0.581754,0.151187,0.115867,0.732941,0.848808,0.286511,0.025697,0.271131,0.024626,0.781577,0.822572,4,5,0.032668,0.008997,3.631048,1.216465e-01,4.987579e-02,2.175245e-01,2.999757,1.459400,8.492593,0.143496,0.034588,0.547003,0.331167,6494,5866,balanced_pixels_center,NaN,NaN,6.142479,4.289968,NaN,NaN,NaN,-0.0,-0.0,-0.272727,6.142479,6.142479,4.289968,6.933929,0.401533,-0.653120,-0.267136,6.472231,7.410044,2.389603,1.852511,0.017752,0.028918,NaN,NaN,3.785244,batch_sensitive,batch_sensitive; score_unstable,False
3,balanced_pixels_random,1,pixel_matrix,random,random,balanced_pixels,snv_sg_smooth,snv+sg_smooth,12360,63,20,40.0,40.0,"{'almond': 6494, 'peanut': 5866}",0.662961,0.114969,0.095564,0.777930,0.873493,0.265611,0.006142,0.205473,0.041886,0.653878,0.728653,4,4,0.017140,0.003037,5.643716,1.745068e-01,1.010641e-01,5.382438e-01,2.999757,0.834746,8.904764,0.192140,0.027821,1.115901,0.118025,6494,5866,balanced_pixels_random,NaN,NaN,6.874581,5.823665,NaN,NaN,NaN,-0.0,-0.0,-0.163636,6.874581,6.874581,5.823665,8.499360,0.027473,-0.970353,-0.518262,6.666354,4.203665,2.496462,1.050916,0.017752,0.028918,NaN,NaN,3.785244,score_unstable,score_unstable,False


In [29]:
# Detailed plots: top candidate globally + best candidate for each matrix variant.
selected_rows = []

selected_rows.append(pca_scored_df.iloc[0])

for _, row in best_by_variant_df.iterrows():
    selected_rows.append(row)

# Deduplicate by run_id + matrix_method + preprocessing
selected_df = pd.DataFrame(selected_rows).drop_duplicates(
    subset=["run_id", "matrix_method", "preprocessing", "balanced_pixel_strategy"]
).reset_index(drop=True)

selected_df = selected_df.head(N_TOP_TO_PLOT)

display(
    selected_df[
        [
            "run_id",
            "matrix_variant",
            "matrix_method",
            "balanced_pixel_strategy",
            "preprocessing",
            "selection_score",
            "selection_flag",
            "class_trace_ratio",
            "batch_trace_ratio",
            "object_class_trace_ratio",
            "object_batch_trace_ratio",
        ]
    ]
)

,run_id,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,class_trace_ratio,batch_trace_ratio,object_class_trace_ratio,object_batch_trace_ratio
0,balanced_pixels_random,balanced_pixels_random,balanced_pixels,random,snv_sg_smooth,5.823665,score_unstable,0.017140,0.003037,0.192140,0.027821
1,object_matrices,object_mean,object_mean,not_applicable,absorbance_sg_d1,2.568244,candidate,0.053277,0.004290,NaN,NaN
2,object_matrices,object_median,object_median,not_applicable,absorbance_sg_d1,1.972732,candidate,0.075748,0.003593,NaN,NaN
3,balanced_pixels_center,balanced_pixels_center,balanced_pixels,center,snv_sg_smooth,4.289968,batch_sensitive,0.032668,0.008997,0.143496,0.034588


In [30]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)
        pca = result["pca"]

        title = (
            f"Explained variance — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_explained_variance(
            pca.explained_variance_ratio_,
            pca.cumulative_explained_variance_ratio_,
            n_components_to_show=min(N_COMPONENTS, 12),
            title=title,
            show=True,
        )
else:
    print("Detailed explained variance plots skipped.")

In [31]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(
            row,
            pca_results_registry,
        )

        pca = result["pca"]
        scores = result["scores"]
        y = result["y"]
        metadata = result["metadata"]

        matrix_method = str(row["matrix_method"])
        is_pixel_matrix = matrix_method in {
            "balanced_pixels",
            "all_pixels",
        }

        title = (
            f"PCA scores PC1/PC2 — "
            f"{row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        if not is_pixel_matrix:
            plot_scores(
                scores,
                dims=(1, 2),
                labels=y,
                color_by="label",
                object_ids=metadata_value(
                    metadata,
                    "object_id",
                    "observation_ids",
                ),
                source_images=metadata_value(
                    metadata,
                    "source_image",
                    "source_images",
                ),
                batches=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                areas=metadata_value(
                    metadata,
                    "area_pixels",
                    "area",
                    "areas",
                ),
                symbol_by="batch",
                category_order=["almond", "peanut"],
                component_variance=pca.explained_variance_ratio_,
                component_prefix="PC",
                title=title,
                show=True,
            )

        else:
            score_df = build_scores_dataframe(
                scores,
                labels=y,
                meta=metadata,
                dims=(1, 2),
                score_prefix="PC",
            )

            plot_scores_density(
                score_df,
                x="PC1",
                y="PC2",
                color_by="label",
                facet_col=(
                    "batch"
                    if "batch" in score_df.columns
                    else None
                ),
                mode="contour",
                title=title,
                show=True,
            )

In [32]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(
            row,
            pca_results_registry,
        )

        pca = result["pca"]
        scores = result["scores"]
        y = result["y"]
        metadata = result["metadata"]

        matrix_method = str(row["matrix_method"])
        is_pixel_matrix = matrix_method in {
            "balanced_pixels",
            "all_pixels",
        }

        title = (
            f"PCA scores PC1/PC2 colored by batch — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        if not is_pixel_matrix:
            plot_scores(
                scores,
                dims=(1, 2),
                labels=y,
                color_values=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                color_by="batch",
                object_ids=metadata_value(
                    metadata,
                    "object_id",
                    "observation_ids",
                ),
                source_images=metadata_value(
                    metadata,
                    "source_image",
                    "source_images",
                ),
                batches=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                areas=metadata_value(
                    metadata,
                    "area_pixels",
                    "area",
                    "areas",
                ),
                symbol_by="batch",
                category_order=["almond", "peanut"],
                component_variance=pca.explained_variance_ratio_,
                component_prefix="PC",
                title=title,
                show=True,
            )
        else:
            score_df = build_scores_dataframe(
                scores,
                labels=y,
                meta=metadata,
                dims=(1, 2),
                score_prefix="PC",
            )

            object_score_df = summarize_scores_by_object(
                score_df,
                score_cols=("PC1", "PC2"),
                object_col="object_id",
                extra_group_cols=[
                    "label",
                    "batch",
                    "source_image",
                    "subset",
                ],
            )

            plot_object_score_summary(
                object_score_df,
                x="PC1_mean",
                y="PC2_mean",
                color_by="label",
                symbol_by="batch",
                facet_col=None,
                size_by="n_pixels",
                title=(
                    f"Object summary of pixel PCA scores — "
                    f"{row['matrix_variant']} — "
                    f"{row['preprocessing']}"
                ),
                show=True,
            )

In [33]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)

        loadings = result["loadings"]

        title = (
            f"PCA loadings — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_loadings(
            loadings,
            wavelengths=wavelengths,
            components=(1, 2, 3),
            explained_variance_ratio=(
                result["pca"].explained_variance_ratio_
            ),
            title=title,
            show=True,
        )
else:
    print("Detailed PCA loadings plots skipped.")

In [34]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)

        pca = result["pca"]
        X_pre = result["X_preprocessed"]
        y = result["y"]
        metadata = result["metadata"]

        object_ids = metadata_value(metadata, "object_id", "observation_ids")
        source_images = metadata_value(metadata, "source_image", "source_images")

        title = (
            f"PCA diagnostic Q vs T² — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_pca_diagnostic(
            pca,
            X=X_pre,
            labels=y,
            object_ids=object_ids,
            source_images=source_images,
            n_components=min(3, N_COMPONENTS),
            title=title,
            show=True,
        )
else:
    print("Detailed PCA diagnostic plots skipped.")

In [35]:
def mean_class_difference_by_preprocessing(
    pca_results_registry,
    target_class="peanut",
    reference_class="almond",
):
    rows = []

    for run_id, results_by_matrix in pca_results_registry.items():
        for matrix_method, results_by_preproc in results_by_matrix.items():
            for preprocessing, res in results_by_preproc.items():
                X = res["X_preprocessed"]
                y = np.asarray(res["y"]).astype(str)

                if target_class not in y or reference_class not in y:
                    continue

                mean_target = X[y == target_class].mean(axis=0)
                mean_reference = X[y == reference_class].mean(axis=0)
                diff = mean_target - mean_reference

                rows.append({
                    "run_id": run_id,
                    "matrix_method": matrix_method,
                    "preprocessing": preprocessing,
                    "mean_abs_difference": float(np.mean(np.abs(diff))),
                    "max_abs_difference": float(np.max(np.abs(diff))),
                    "wavelength_max_difference": float(wavelengths[np.argmax(np.abs(diff))]),
                })

    return pd.DataFrame(rows)


class_difference_df = mean_class_difference_by_preprocessing(
    pca_results_registry,
    target_class=TARGET_CLASS,
    reference_class="almond",
)

display(
    class_difference_df
    .sort_values("mean_abs_difference", ascending=False)
    .head(20)
)

,run_id,matrix_method,preprocessing,mean_abs_difference,max_abs_difference,wavelength_max_difference
33,object_matrices,object_median,absorbance_snv_sg_smooth,0.040696,0.201459,1702.000000
25,object_matrices,object_median,absorbance_snv,0.040019,0.244538,1702.000000
15,object_matrices,object_mean,absorbance_snv_sg_smooth,0.039340,0.159603,1702.000000
7,object_matrices,object_mean,absorbance_snv,0.038850,0.174617,1702.000000
69,balanced_pixels_center,balanced_pixels,absorbance_snv_sg_smooth,0.038554,0.147064,1702.000000
61,balanced_pixels_center,balanced_pixels,absorbance_snv,0.038097,0.155349,1702.000000
51,balanced_pixels_random,balanced_pixels,absorbance_snv_sg_smooth,0.037651,0.149089,1438.970588
43,balanced_pixels_random,balanced_pixels,absorbance_snv,0.037418,0.150122,1415.058824
30,object_matrices,object_median,snv_sg_smooth,0.037179,0.175933,1702.000000
20,object_matrices,object_median,snv,0.037123,0.201962,1702.000000


## 3. Interpretation table

We prepare a compact interpretation table to help select candidate preprocessing methods.

For PCA/MCR exploration, good candidates should generally have:

- strong class separation,
- limited batch effect,
- interpretable loadings,
- stable spectra after preprocessing,
- no excessive loss of spectral structure.

In [36]:
interpretation_cols = [
    "matrix_variant",
    "preprocessing",
    "selection_score",
    "selection_flag",
    "n_observations",
    "ncomp_90",
    "ncomp_95",
    "cum_pc3",
    "class_trace_ratio",
    "batch_trace_ratio",
    "class_over_batch_ratio",
    "fisher_pc1",
    "fisher_pc2",
    "mahalanobis_pc1_pc2_pc3",
    "object_class_trace_ratio",
    "object_batch_trace_ratio",
    "object_over_intra_ratio",
    "mean_intra_object_trace",
]

available_cols = [col for col in interpretation_cols if col in pca_scored_df.columns]

# interpretation_df = pca_scored_df[available_cols].copy()

# interpretation_df

In [37]:
# interpretation_objects_df = interpretation_df[interpretation_df['matrix_variant'].str.startswith('object')]
# interpretation_objects_df

In [38]:
# interpretation_pixels_df = interpretation_df[~interpretation_df['matrix_variant'].str.startswith('object')]
# interpretation_pixels_df

In [39]:
# preproc_objects = interpretation_objects_df[interpretation_objects_df['selection_score']>2.5]['preprocessing'].unique().tolist()
# preproc_objects

In [40]:
# preproc_pixels = interpretation_pixels_df[interpretation_pixels_df['selection_score']>3]['preprocessing'].unique().tolist()
# preproc_pixels

In [41]:
# preproc_study = set(preproc_pixels + preproc_objects)
# preproc_study

In [42]:
# shortlist_df = pca_scored_df[
#     pca_scored_df["preprocessing"].isin(preproc_study)
# ].copy()

# shortlist_df = (
#     shortlist_df
#     .sort_values("selection_score", ascending=False)
#     .groupby("matrix_variant", group_keys=False)
#     .head(5)
#     .reset_index(drop=True)
# )

# shortlist_df[available_cols]

In [43]:
pca_selected_preprocessings_df, candidate_pool_df, selection_family_counts = (
    select_pca_preprocessing_shortlist(
        pca_scored_df,
        config=PCA_SELECTION_CONFIG,
    )
)

selection_display_cols = [
    "matrix_family",
    "family_selection_rank",
    "preprocessing",
    "preprocessing_steps",
    "selection_score",
    "selection_score_without_stability",
    "selection_score_stability_std",
    "rank",
    "matrix_variant",
    "selection_reason",
    "selection_flag",
    "pca_validation_warning",
    "matrix_method",
    "balanced_pixel_strategy",
]
selection_display_cols = [
    col for col in selection_display_cols
    if col in pca_selected_preprocessings_df.columns
]

candidate_display_cols = [
    col for col in [
        *available_ranking_cols,
        "selection_score_without_stability",
        "selection_score_stability_std",
        "selection_flag",
        "pca_validation_warning",
    ]
    if col in candidate_pool_df.columns
]

print("Strict PCA preprocessing selection by matrix family:")
print(selection_family_counts.to_string())

display(candidate_pool_df[candidate_display_cols].head(30))
display(pca_selected_preprocessings_df[selection_display_cols])

print("Selected preprocessing names by matrix family:")
for family, group in pca_selected_preprocessings_df.groupby("matrix_family", sort=True):
    print(f" - {family}: {sorted(group['preprocessing'].unique())}")


Strict PCA preprocessing selection by matrix family:
matrix_family
object_matrix    5
pixel_matrix     5


,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace,selection_score_without_stability,selection_score_stability_std,selection_flag,pca_validation_warning
0,35,object_matrix,object_mean,object_mean,not_applicable,absorbance,0.668315,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.955463,0.995853,1,1,0.084129,0.064563,0.568978,0.041420,0.019906,2.080797,NaN,NaN,NaN,NaN,0.912198,0.975531,candidate,
1,41,object_matrix,object_mean,object_mean,not_applicable,absorbance_msc,0.411133,batch_sensitive,317,166,151,"{'almond': 166, 'peanut': 151}",0.518238,0.885179,4,5,0.256717,2.150329,4.442265,0.330485,0.032187,10.267793,NaN,NaN,NaN,NaN,1.987057,6.303696,batch_sensitive,batch_sensitive; score_unstable
2,9,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,2.568244,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.769020,0.965067,2,3,0.037632,0.448945,0.663760,0.053277,0.004290,12.418964,NaN,NaN,NaN,NaN,2.896111,1.311469,candidate,
3,15,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d2,2.169377,weak_relative_separation,317,166,151,"{'almond': 166, 'peanut': 151}",0.860111,0.973547,2,3,0.010468,0.445031,0.024193,0.028776,0.003217,8.944332,NaN,NaN,NaN,NaN,2.500842,1.325858,weak_relative_separation,weak_relative_separation
4,36,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_smooth,0.665915,weak_relative_separation,317,166,151,"{'almond': 166, 'peanut': 151}",0.956767,0.996478,1,1,0.084029,0.063787,0.566963,0.041379,0.019915,2.077757,NaN,NaN,NaN,NaN,0.909810,0.975580,weak_relative_separation,weak_relative_separation
5,39,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv,0.414930,batch_sensitive,317,166,151,"{'almond': 166, 'peanut': 151}",0.517074,0.884388,4,5,0.257517,2.156342,4.465663,0.331451,0.032249,10.277995,NaN,NaN,NaN,NaN,1.994003,6.316291,batch_sensitive,batch_sensitive; score_unstable
6,32,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_d1,0.932149,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.833440,0.966479,2,3,0.149999,0.925057,3.450543,0.122692,0.021761,5.638177,NaN,NaN,NaN,NaN,1.647698,2.862198,candidate,
7,18,object_matrix,object_median,object_median,not_applicable,absorbance_snv_sg_d2,1.964249,candidate,317,166,151,"{'almond': 166, 'peanut': 151}",0.884374,0.950063,2,3,0.215445,0.043348,0.364598,0.100973,0.005256,19.209220,NaN,NaN,NaN,NaN,2.548774,2.338102,candidate,
8,33,object_matrix,object_median,object_median,not_applicable,absorbance_snv_sg_smooth,0.827933,score_unstable,317,166,151,"{'almond': 166, 'peanut': 151}",0.515113,0.859508,4,5,0.390359,0.988409,2.826110,0.267170,0.024087,11.092096,NaN,NaN,NaN,NaN,1.849466,4.086130,score_unstable,score_unstable
9,47,object_matrix,object_mean,object_mean,not_applicable,msc,0.122879,batch_sensitive,317,166,151,"{'almond': 166, 'peanut': 151}",0.422694,0.878011,4,5,0.503175,0.511812,4.669004,0.305316,0.032944,9.267822,NaN,NaN,NaN,NaN,1.645766,6.091545,batch_sensitive,batch_sensitive; score_unstable


,matrix_family,family_selection_rank,preprocessing,preprocessing_steps,selection_score,selection_score_without_stability,selection_score_stability_std,rank,matrix_variant,selection_reason,selection_flag,pca_validation_warning,matrix_method,balanced_pixel_strategy
0,object_matrix,1,absorbance_sg_d1,absorbance+sg_d1,2.568244,2.896111,1.311469,9,object_mean,top_5_preprocessing_within_object_matrix; best...,candidate,,object_mean,not_applicable
1,object_matrix,2,absorbance_sg_d2,absorbance+sg_d2,2.169377,2.500842,1.325858,15,object_mean,top_5_preprocessing_within_object_matrix; best...,weak_relative_separation,weak_relative_separation,object_mean,not_applicable
2,object_matrix,3,absorbance_snv_sg_d2,absorbance+snv+sg_d2,1.964249,2.548774,2.338102,18,object_median,top_5_preprocessing_within_object_matrix; best...,candidate,,object_median,not_applicable
3,object_matrix,4,snv_sg_d2,snv+sg_d2,1.300178,1.764206,1.856110,28,object_median,top_5_preprocessing_within_object_matrix; best...,candidate,,object_median,not_applicable
4,object_matrix,5,absorbance_snv_sg_d1,absorbance+snv+sg_d1,0.932149,1.647698,2.862198,32,object_mean,top_5_preprocessing_within_object_matrix; best...,candidate,,object_mean,not_applicable
5,pixel_matrix,1,snv_sg_smooth,snv+sg_smooth,5.823665,6.874581,4.203665,1,balanced_pixels_random,top_5_preprocessing_within_pixel_matrix; best_...,score_unstable,score_unstable,balanced_pixels,random
6,pixel_matrix,2,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,5.254788,6.143726,3.555753,2,balanced_pixels_random,top_5_preprocessing_within_pixel_matrix; best_...,candidate,,balanced_pixels,random
7,pixel_matrix,3,snv,snv,5.047769,6.023611,3.903368,3,balanced_pixels_random,top_5_preprocessing_within_pixel_matrix; best_...,score_unstable,score_unstable,balanced_pixels,random
8,pixel_matrix,4,sg_smooth,sg_smooth,2.797507,3.126480,1.315891,6,balanced_pixels_center,top_5_preprocessing_within_pixel_matrix; best_...,candidate,,balanced_pixels,center
9,pixel_matrix,5,raw,raw,2.794307,3.123342,1.316140,7,balanced_pixels_center,top_5_preprocessing_within_pixel_matrix; best_...,candidate,,balanced_pixels,center


Selected preprocessing names by matrix family:
 - object_matrix: ['absorbance_sg_d1', 'absorbance_sg_d2', 'absorbance_snv_sg_d1', 'absorbance_snv_sg_d2', 'snv_sg_d2']
 - pixel_matrix: ['absorbance_snv_sg_smooth', 'raw', 'sg_smooth', 'snv', 'snv_sg_smooth']


In [44]:
validate_pca_preprocessing_shortlist(
    pca_selected_preprocessings_df,
    max_per_family=MAX_PREPROCESSINGS_PER_MATRIX_FAMILY,
    expected_families=EXPECTED_PCA_MATRIX_FAMILIES,
    context="before parquet save",
)

save_parquet(
    pca_selected_preprocessings_df,
    PCA_SELECTED_PREPROCESSINGS_PATH,
)

saved_pca_selected_preprocessings_df = pd.read_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
selection_family_counts = validate_pca_preprocessing_shortlist(
    saved_pca_selected_preprocessings_df,
    max_per_family=MAX_PREPROCESSINGS_PER_MATRIX_FAMILY,
    expected_families=EXPECTED_PCA_MATRIX_FAMILIES,
    context="after parquet save",
)

print("Saved selected PCA preprocessings:")
print(" -", PCA_SELECTED_PREPROCESSINGS_PATH)
print("Rows by matrix family:")
print(selection_family_counts.to_string())


Saved selected PCA preprocessings:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
Rows by matrix family:
matrix_family
object_matrix    5
pixel_matrix     5


In [45]:
pca_protocol = {
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)) if wavelengths is not None else np.nan,

    "target_class": TARGET_CLASS,
    "reference_classes": list(REFERENCE_CLASSES),
    "pca_sample_kind": PCA_SAMPLE_KIND,
    "pca_allowed_batches": PCA_ALLOWED_BATCHES,

    "n_components": int(N_COMPONENTS),
    "m_balanced_pixels": int(M_BALANCED_PIXELS),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "random_state": int(RANDOM_STATE),
    "balanced_pixel_strategies": BALANCED_PIXEL_STRATEGIES,
    "run_all_pixels": bool(RUN_ALL_PIXELS),

    "sg_window_length": int(SG_WINDOW_LENGTH),
    "sg_polyorder": int(SG_POLYORDER),

    "pca_selection_config_source": "src.experiment_config.make_pca_selection_config",
    "max_preprocessings_per_matrix_family": int(MAX_PREPROCESSINGS_PER_MATRIX_FAMILY),
    "expected_pca_matrix_families": ",".join(EXPECTED_PCA_MATRIX_FAMILIES),
    "pca_selection_profile_families": ",".join(PCA_SELECTION_CONFIG.profiles.keys()),
    "pca_selection_group_cols": ",".join(PCA_SELECTION_CONFIG.group_cols),
    "pca_selection_robust_scaling": bool(PCA_SELECTION_CONFIG.robust),
    "pca_selection_clip_quantiles": ",".join(str(value) for value in PCA_SELECTION_CONFIG.clip_quantiles),
    "pca_selection_bootstrap_iterations": int(PCA_SELECTION_CONFIG.stability_bootstrap_iterations),
    "pca_selection_stability_penalty_weight": float(PCA_SELECTION_CONFIG.stability_penalty_weight),
    "pca_selection_quality_lower_quantile": float(PCA_SELECTION_CONFIG.quality_lower_quantile),
    "pca_selection_quality_upper_quantile": float(PCA_SELECTION_CONFIG.quality_upper_quantile),
    "pca_selection_validation_upper_quantile": float(PCA_SELECTION_CONFIG.validation_upper_quantile),
    "selected_rows_by_matrix_family": "; ".join(
        f"{family}={count}"
        for family, count in selection_family_counts.items()
    ),

    "n_pca_combinations": int(len(pca_scored_df)),
    "n_selected_preprocessing_rows": int(len(pca_selected_preprocessings_df)),
    "n_selected_preprocessing_names": int(pca_selected_preprocessings_df["preprocessing"].nunique()),

    "pca_summary_path": str(PCA_SUMMARY_PATH),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
}

pca_protocol_df = pd.DataFrame([pca_protocol])

print("PCA protocol summary:")
display(pca_protocol_df)

PCA protocol summary:


,db_h5_path,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,reference_classes,pca_sample_kind,pca_allowed_batches,n_components,m_balanced_pixels,replace_balanced_pixels,random_state,balanced_pixel_strategies,run_all_pixels,sg_window_length,sg_polyorder,pca_selection_config_source,max_preprocessings_per_matrix_family,expected_pca_matrix_families,pca_selection_profile_families,pca_selection_group_cols,pca_selection_robust_scaling,pca_selection_clip_quantiles,pca_selection_bootstrap_iterations,pca_selection_stability_penalty_weight,pca_selection_quality_lower_quantile,pca_selection_quality_upper_quantile,pca_selection_validation_upper_quantile,selected_rows_by_matrix_family,n_pca_combinations,n_selected_preprocessing_rows,n_selected_preprocessing_names,pca_summary_path,pca_selected_preprocessings_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,"[almond, peanut]",pure,"[1, 2, 3]",20,40,False,42,"[random, center]",False,11,2,src.experiment_config.make_pca_selection_config,5,"object_matrix,pixel_matrix","object_matrix,pixel_matrix",matrix_variant,True,"0.05,0.95",100,0.25,0.25,0.75,0.75,object_matrix=5; pixel_matrix=5,72,10,10,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [46]:
print("03_pca_exploration_selection.ipynb completed.")
print()
print("Essential outputs:")
print(" -", PCA_SUMMARY_PATH)
print(" -", PCA_SCORING_DIAGNOSTICS_PATH)
print(" -", PCA_SELECTED_PREPROCESSINGS_PATH)
print()
print("Summary:")
print(f" - Wavelength mode: {WAVELENGTH_MODE}")
print(f" - Active bands: {len(wavelengths) if wavelengths is not None else 'unknown'}")
print(f" - PCA object subset size: {len(object_db_pca)}")
print(f" - Valid preprocessing methods: {len(valid_preprocessing_configs)}")
print(f" - PCA combinations evaluated: {len(pca_scored_df)}")
print(f" - Selected preprocessing rows: {len(pca_selected_preprocessings_df)}")
print(f" - Selected preprocessing names: {pca_selected_preprocessings_df['preprocessing'].nunique()}")
print()
print("Best global candidate:")
display(pca_scored_df.head(1)[available_cols])
print()
print("Next notebook:")
print("04A_simca_grid_validation.ipynb")

03_pca_exploration_selection.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_scoring_diagnostics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet

Summary:
 - Wavelength mode: non_noisy_all
 - Active bands: 63
 - PCA object subset size: 317
 - Valid preprocessing methods: 18
 - PCA combinations evaluated: 72
 - Selected preprocessing rows: 10
 - Selected preprocessing names: 10

Best global candidate:


,matrix_variant,preprocessing,selection_score,selection_flag,n_observations,ncomp_90,ncomp_95,cum_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,balanced_pixels_random,snv_sg_smooth,5.823665,score_unstable,12360,4,4,0.873493,0.01714,0.003037,5.643716,0.006142,0.205473,0.728653,0.19214,0.027821,0.118025,1.115901



Next notebook:
04A_simca_grid_validation.ipynb
